> ## SOLUTION / ANSWER KEY &mdash; Lab 3.11
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-11-feature-extraction-head.ipynb`](../lab-11-feature-extraction-head.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 3.11 &mdash; Feature Extraction + a Classifier Head

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 40 min &nbsp;|&nbsp; **Day 2 &middot; Module 3 &mdash; Why Transformers?**

### What you'll do
- Turn text into real feature vectors with a frozen model
- Train a lightweight classifier head on those features
- Predict sentiment on unseen sentences

> **How this lab works (near-real):** these labs run **real Hugging Face Transformers** locally on CPU. Read the **Concept**, fill the real `___` blanks in **Build it** (real tokenizer / model / decoding calls), **Run it for real** to see the **actual model output**, note **What to notice**, then finish with an open **Your turn**. There is **no auto-grader** &mdash; the goal is real model output you can read. The genuine maths (attention, positional encoding, cosine) you still compute **by hand** &mdash; that is real mechanics, not a stub.

> **Models:** small, CPU-friendly models from the HF hub &mdash; `distilbert-base-uncased` (tokenizer / fill-mask / attention), `sentence-transformers/all-MiniLM-L6-v2` (embeddings), `distilgpt2` (generation). First use downloads the weights (needs network), then they are cached. The hosted "GPT API" path uses `ChatGroq` (`GROQ_API_KEY` in `.env`).

**Reference:** [Module 3 slides &mdash; The model landscape](../../presentation/day2-module3-why-transformers.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 3 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, pathlib
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("USE_TF", "0")                 # these labs are torch-only; skip the TF backend
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")   # mute TensorFlow's C++ INFO/WARNING startup noise
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True), override=True)   # GROQ_API_KEY etc. (used by the text-gen lab)

WORK = os.path.join(os.environ.get("TEMP") or os.environ.get("TMP") or "/tmp", "biaa-lab-03-11")
os.makedirs(WORK, exist_ok=True)
print("WORK:", WORK)
print("Real Hugging Face models load from the hub on first use (one-time download, then cached).")

## Concept
A powerful pattern: let a big model turn text into **features** (embeddings), freeze it, then train a
tiny **head** on top for your task &mdash; **transfer learning** (the heart of Module 4). We use the
**real** all-MiniLM-L6-v2 as the frozen feature extractor and fit a logistic-regression head for
sentiment. A handful of examples is enough because the features already carry meaning.

> Needs `scikit-learn` (already in the lab venv).

## Build it
`embed()` (given) is the real feature extractor. Train the head and complete `predict`.

In [ ]:
import numpy as np
_EMB = {}
def embed(texts):
    """Real sentence embeddings from all-MiniLM-L6-v2: model forward pass -> mean-pool -> unit vector."""
    import torch
    from transformers import AutoTokenizer, AutoModel
    if not _EMB:
        name = "sentence-transformers/all-MiniLM-L6-v2"
        _EMB["tok"] = AutoTokenizer.from_pretrained(name)
        _EMB["mdl"] = AutoModel.from_pretrained(name); _EMB["mdl"].eval()
    if isinstance(texts, str): texts = [texts]
    enc = _EMB["tok"](texts, padding=True, truncation=True, return_tensors="pt")
    with torch.no_grad(): out = _EMB["mdl"](**enc)
    mask = enc["attention_mask"].unsqueeze(-1).float()
    pooled = (out.last_hidden_state * mask).sum(1) / mask.sum(1)     # mean over REAL tokens
    return torch.nn.functional.normalize(pooled, dim=1).numpy()      # unit vectors -> dot == cosine

from sklearn.linear_model import LogisticRegression
TRAIN = [("i love this movie", 1), ("what a great film", 1), ("wonderful and amazing", 1),
         ("brilliant acting throughout", 1), ("an absolute delight to watch", 1),
         ("i hate this movie", 0), ("what an awful film", 0), ("terrible and boring", 0),
         ("bad acting throughout", 0), ("a complete waste of time", 0)]

def train_head(X, y):
    return LogisticRegression(max_iter=1000).fit(X, y)

def predict(clf, text):
    return int(clf.predict(embed([text]))[0])

## Run it for real
Extract real features, train the head, and classify unseen sentences.

In [ ]:
try:
    texts = [t for t, _ in TRAIN]; labels = [y for _, y in TRAIN]
    X = embed(texts)                       # real frozen feature extractor -> (n, 384)
    print("feature matrix shape:", X.shape)
    clf = train_head(X, labels)            # train ONLY the small head
    for s in ["i really enjoyed it", "an absolute disaster", "the plot was dull and lifeless",
              "a heartwarming and beautiful story"]:
        print(f"  pred={predict(clf, s)}  <-  {s}")
except Exception as e:
    print("(If a ___ is still unfilled, fill it above. On first run the model downloads")
    print(" from the Hugging Face hub -- that needs network; re-run once it finishes.)")
    print("  reason:", type(e).__name__, "--", e)

## What to notice
- The head trains on **10 examples** and still generalises &mdash; because the **frozen model's features** already encode sentiment.
- You never touched the transformer's weights: **feature extraction = frozen backbone + a trained head**. Cheap, fast, no GPU.
- Swap the head (SVM, small MLP) or the task (topic, spam) &mdash; the same embeddings power all of them.

## Your turn (open task &mdash; no grader)
Add your own labelled sentences (try a **neutral** class for 3-way). Find a test sentence the
head gets **wrong** &mdash; is it a feature limitation or too little training data? A "good" answer:
you can articulate why transfer learning needs so few examples here.

In [ ]:
# --- Reference answer (ONE good way to do the 'Your turn' task -- compare with your own) ---
try:
    extra = [("it was okay nothing special", 0), ("a masterpiece i will rewatch", 1),
             ("mediocre but watchable", 0), ("stunning from start to finish", 1)]
    texts = [t for t, _ in TRAIN + extra]; labels = [y for _, y in TRAIN + extra]
    clf2 = train_head(embed(texts), labels)       # retrain the head on more examples
    for s in ["surprisingly enjoyable", "painfully boring", "a real triumph"]:
        print(f"  pred={predict(clf2, s)}  <-  {s}")   # few examples suffice: features already carry meaning
except Exception as e:
    print("(If a ___ is still unfilled, fill it above. On first run the model downloads")
    print(" from the Hugging Face hub -- that needs network; re-run once it finishes.)")
    print("  reason:", type(e).__name__, "--", e)

---
### Key takeaway
Frozen features from a real model + a tiny trained head = transfer learning. Module 4 goes one step further and fine-tunes the backbone itself.

[&#8592; All Module 3 labs](./index.html) &nbsp;&middot;&nbsp; [Module 3 slides](../../presentation/day2-module3-why-transformers.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>